In [1]:
import { PGlite } from "npm:@electric-sql/pglite";
import { pg_trgm } from "npm:@electric-sql/pglite/contrib/pg_trgm";
import { unaccent } from "npm:@electric-sql/pglite/contrib/unaccent";

const client = new PGlite(
  process.env.test ? "memory://" : "./../../web/.data/database",
  {
    extensions: { pg_trgm, unaccent },
  },
);


In [2]:
import { cert, getApp, getApps, initializeApp } from "npm:firebase-admin/app";
import { getFirestore } from "npm:firebase-admin/firestore";

const serviceAccountText = await Deno.readTextFile("./../env/firebase.json");
const serviceAccount = JSON.parse(serviceAccountText);

const app = getApps().length === 0
  ? initializeApp({ credential: cert(serviceAccount) })
  : getApp();

const firestore = getFirestore(app);


In [3]:
const clubsCollection = firestore.collection("clubs");
const clubsSnapshot = await clubsCollection.get();


In [4]:
import { stringify } from "jsr:@std/csv";

const clubsData = clubsSnapshot.docs.map((doc) => ({
  uid: doc.id,
  name: doc.data().name,
  isActive: doc.data().active,
  league: doc.data().zdp ? "junior" : "senior",
  region: null,
}));

const csvData = stringify(clubsData, {
  columns: ["uid", "name", "isActive", "league", "region"],
});

await Deno.mkdir("./../data", { recursive: true });
await Deno.writeTextFile("./../data/clubs.csv", csvData);


In [5]:
const usersCollection = firestore.collection("users");
// Get the data sorted by createdAt in ascending order
const usersSnapshot = await usersCollection.orderBy("createdAt", "asc").get();


In [6]:
import { stringify } from "jsr:@std/csv";

const usersData = usersSnapshot.docs.map((doc) => {
  const data = doc.data();

  return {
    uid: doc.id,
    name: data.name ?? "",
    surname: data.surname ?? "",
    role: data.role ?? "user",
    email: data.email ?? "",
    phone: data.phone ?? "",
    birthDate: data.birthdate?.toDate()?.toISOString().split("T")[0] ?? "",
    createdAt: data.createdAt?.toDate()?.toISOString() ?? "",
    clubId: data.club?.id ?? "",
    seasons: data.seasons
      ? data.seasons.map((season: { year: number | string }) => season.year)
        .join(";")
      : "",
    clubManager: data.clubManager ?? false,
    address: data.address ?? "",
    streetAddress: "",
    postalCode: data.postalCode ?? "",
    city: data.city ?? "",
  };
});

const usersCsvData = stringify(usersData, {
  columns: [
    "uid",
    "name",
    "surname",
    "role",
    "email",
    "phone",
    "birthDate",
    "createdAt",
    "clubId",
    "seasons",
    "clubManager",
    "address",
    "streetAddress",
    "postalCode",
    "city",
  ],
});

await Deno.writeTextFile("./../data/users.csv", usersCsvData);


In [7]:
const hashesCollection = firestore.collection("hashes");
const hashesSnapshot = await hashesCollection.get();


In [8]:
// Go through each document in the hashes collection and if the group them by the email field. Keep only the latest document for each email (createdAt field).

const hashesData = hashesSnapshot.docs.map((doc) => ({
  uid: doc.id,
  email: doc.data().email ?? "",
  hash: doc.data().hash ?? "",
  createdAt: doc.data().createdAt?.toDate()?.toISOString() ?? "",
}));

const latestHashesMap = new Map<string, any>();

for (const hash of hashesData) {
  const existingHash = latestHashesMap.get(hash.email);

  if (
    !existingHash || new Date(hash.createdAt) > new Date(existingHash.createdAt)
  ) {
    latestHashesMap.set(hash.email, hash);
  }
}

const latestHashesData = Array.from(latestHashesMap.values());


In [9]:
// Read the file clubs_x.csv
import { parse } from "jsr:@std/csv";

type ClubX = {
  id: number | null;
  uid: string;
  name: string;
  isActive: boolean;
  league: "junior" | "senior" | "university" | null;
  region: "western" | "central" | "eastern" | null;
};

const clubsXFile = await Deno.readTextFile("./../data/clubs_x.csv");
const clubsXData: ClubX[] = await parse(clubsXFile, {
  skipFirstRow: true,
  strip: true,
  columns: ["uid", "name", "isActive", "league", "region"],
});

// Add the clubxdata to the pglite client and map generated ids to the clubs x  using returning data.
clubsXData.forEach((club) => {
  client.query(
    "INSERT INTO clubs (name, is_active, league, region) VALUES ($1, $2, $3, $4) RETURNING id",
    [club.name, club.isActive, club.league, club.region],
  ).then((result) => {
    const generatedId = result.rows[0].id;
    clubsXData.find((c) => c.uid === club.uid)!.id = generatedId;
    console.log(`Inserted club ${club.name} with generated id ${generatedId}`);
  }).catch((error) => {
    console.error(`Error inserting club ${club.name}:`, error);
  });
});


Inserted club Sučany with generated id 1
Inserted club SZŠ Félix v Źiline with generated id 2
Inserted club Univerzita sv. Cyrila a Metoda v Trnave with generated id 3
Inserted club Súkromná spojená škola, M.Falešníka with generated id 4
Inserted club ZŠ Bartolomeja Krpelca, Bardejov with generated id 5
Inserted club Gym. Varšavská cesta (mladší) with generated id 6
Inserted club ZŠ Martinská, Žilina with generated id 7
Inserted club ŠPMNDaG (mladší) with generated id 8
Inserted club Súkromné gym. Katkin Park with generated id 9
Inserted club Bratislavský debatný spolok with generated id 10
Inserted club Základná Škola u Filipa with generated id 11
Inserted club Marguškin Andrejov a Jozefov Absolventský Klub with generated id 12
Inserted club Gymnázium Andreja Sládkoviča with generated id 13
Inserted club Gym. J. G. Tajovského (Taják) with generated id 14
Inserted club Gym. Leonarda Stockela with generated id 15
Inserted club SPŠE Prešov with generated id 16
Inserted club OA a SOŠ obch

In [ ]:
type UserX = {
  id: number | null;
  uid: string;
  name: string;
  surname: string;
  role: "user" | "admin" | "superadmin" | null;
  email: string;
  phone: string;
  birthDate: string | null;
  createdAt: string | null;
  clubId: number | null;
  seasons: string | null;
  clubManager: boolean | null;
  address: string | null;
  streetAddress: string | null;
  postalCode: string | null;
  city: string | null;
};

// const usersXData: UserX[] = await parse("./../data/users_x.csv");
const usersXFile = await Deno.readTextFile("./../data/users.csv");
const usersXData: UserX[] = await parse(usersXFile, {
  skipFirstRow: true,
  strip: true,
  columns: [
    "uid",
    "name",
    "surname",
    "role",
    "email",
    "phone",
    "birthDate",
    "createdAt",
    "clubId",
    "seasons",
    "clubManager",
    "address",
    "streetAddress",
    "postalCode",
    "city",
  ],
});

// Go trough each user in the usersXData and if the user has role "coach", change it to "user" and set the clubManager field to true.
usersXData.forEach((user) => {
  if (user.role === "coach") {
    user.role = "user";
    user.clubManager = true;
  }
});

// In usersXData, replace all the empty strings with null values.
usersXData.forEach((user) => {
  Object.keys(user).forEach((key) => {
    if (user[key as keyof UserX] === "") {
      user[key as keyof UserX] = null;
    }
  });
});

const removedEmails = new Set<string>();

// Remove all users from usersXData that have name or surname or email as null.
usersXData.forEach((user, index) => {
  if (!user.name || !user.surname || !user.email) {
    console.log(
      `Removing user with uid ${user.uid} because of missing name, surname or email`,
    );
    removedEmails.add(user.email);
    usersXData.splice(index, 1);
  }
});

// Remove all users from usersXData that have birthdate and address as null.
usersXData.forEach((user, index) => {
  if (!user.birthDate && !user.address) {
    console.log(
      `Removing user with uid ${user.uid} because of missing birthdate and address`,
    );
    removedEmails.add(user.email);
    usersXData.splice(index, 1);
  }
});

// Write the removed emails to a file removed_emails.txt
await Deno.writeTextFile(
  "./../data/removed_emails.txt",
  Array.from(removedEmails).join("\n"),
);

// Add the usersxdata to the pglite client and map generated ids to the users x using returning data.
usersXData.forEach((user) => {
  let generatedId: number | null = null;

  client.query(
    "INSERT INTO users (name, surname, role, email, phone, birth_date, created_at, street, postal_code, town, email_verified) VALUES ($1, $2, $3, $4, $5, $6, $7, $8, $9, $10, true) RETURNING id",
    [
      user.name,
      user.surname,
      user.role,
      user.email,
      user.phone,
      user.birthDate,
      user.createdAt,
      user.streetAddress,
      user.postalCode,
      user.city,
    ],
  ).then((result) => {
    generatedId = result.rows[0].id;
    usersXData.find((u) => u.uid === user.uid)!.id = generatedId;
    console.log(
      `Inserted user ${user.name} ${user.surname} with generated id ${generatedId}`,
    );
  }).catch((error) => {
    generatedId = null;
    console.error(`Error inserting user ${user.name} ${user.surname}:`, error);
  });

  // If the user has a clubId, for each season separated by ;, insert a row into the club_memberships table with the generated user id, the club id, and the season year.
  if (user.clubId && user.seasons && generatedId !== null) {
    const seasons = user.seasons.split(";");
    seasons.forEach((season) => {
      mappedClubId = clubsXData.find((c) => c.uid === user.clubId)?.id;
      if (!mappedClubId) {
        console.error(
          `Error: Could not find mapped club id for user ${user.name} ${user.surname} with club uid ${user.clubId}`,
        );
        return;
      }

      // If user is younger than 14 years old in the given season, set the registration_type to "junior_student", < 19 "senior)student", < 30 "graduate" >= 30 "teacher"
      const birthYear = new Date(user.birthDate ?? "").getFullYear();
      const seasonYear = parseInt(season);
      let registrationType:
        | "junior_student"
        | "senior_student"
        | "graduate"
        | "teacher" = "teacher";

      if (birthYear && seasonYear) {
        const age = seasonYear - birthYear;
        if (age < 14) {
          registrationType = "junior_student";
        } else if (age < 19) {
          registrationType = "senior_student";
        } else if (age < 30) {
          registrationType = "graduate";
        } else {
          registrationType = "teacher";
        }
      }

      client.query(
        "INSERT INTO club_memberships (user_id, club_id, season, confirmed, registration_type) VALUES ($1, $2, $3, true, $4)",
        [generatedId, mappedClubId, season, registrationType],
      ).then(() => {
        console.log(
          `Inserted club membership for user ${user.name} ${user.surname} in club ${user.clubId} for season ${season}`,
        );
      }).catch((error) => {
        console.error(
          `Error inserting club membership for user ${user.name} ${user.surname} in club ${user.clubId} for season ${season}:`,
          error,
        );
      });
    });
  }

  // If clubManager is true, insert a row into the club_managers table with the generated user id and the mapped club id.
  if (user.clubManager && user.clubId && generatedId !== null) {
    const mappedClubId = clubsXData.find((c) => c.uid === user.clubId)?.id;
    if (!mappedClubId) {
      console.error(
        `Error: Could not find mapped club id for user ${user.name} ${user.surname} with club uid ${user.clubId}`,
      );
      return;
    }
    client.query(
      "INSERT INTO club_managers (user_id, club_id) VALUES ($1, $2)",
      [generatedId, mappedClubId],
    ).then(() => {
      console.log(
        `Inserted club manager for user ${user.name} ${user.surname} in club ${user.clubId}`,
      );
    }).catch((error) => {
      console.error(
        `Error inserting club manager for user ${user.name} ${user.surname} in club ${user.clubId}:`,
        error,
      );
    });
  }

  // If the user has a hash, insert a row into the accounts table with the generated user id and the hash.
  const userHash = latestHashesData.find((h) => h.email === user.email);
  if (userHash && generatedId !== null) {
    client.query(
      "INSERT INTO accounts (user_id, provider_id, password) VALUES ($1, 'password', $2)",
      [generatedId, userHash.hash],
    ).then(() => {
      console.log(
        `Inserted account for user ${user.name} ${user.surname}`,
      );
    }).catch((error) => {
      console.error(
        `Error inserting account for user ${user.name} ${user.surname}:`,
        error,
      );
    });
  }

  // In usersSnapshot, check if user has supervisor and supervisorEmail fields. If so, insert a row into the legal_guardians table with the generated user id, the supervisor name, and the supervisor email.
  const userSnapshot = usersSnapshot.docs.find((doc) => doc.id === user.uid);
  if (userSnapshot) {
    const userData = userSnapshot.data();
    if (
      userData.supervisor && userData.supervisorEmail && generatedId !== null
    ) {
      client.query(
        "INSERT INTO legal_guardians (user_id, name, email) VALUES ($1, $2, $3)",
        [generatedId, userData.supervisor, userData.supervisorEmail],
      ).then(() => {
        console.log(
          `Inserted legal guardian for user ${user.name} ${user.surname}`,
        );
      }).catch((error) => {
        console.error(
          `Error inserting legal guardian for user ${user.name} ${user.surname}:`,
          error,
        );
      });
    }
  }
});


Inserted user Branislav Juhás with generated id 1
Inserted user Barbora Svitková with generated id 2
Inserted user Riaditeľ / riaditeľka SDA with generated id 3
Inserted user Ondrej Schütz with generated id 4
Inserted user Ondrej Schütz with generated id 5
Inserted user Ondrej Schütz with generated id 6
Inserted user Emma Lujza Barilová with generated id 7
Inserted user Richard Vaško with generated id 8
Inserted user Jakub Bohuš with generated id 9
Inserted user Diana Bulganová with generated id 10
Inserted user Jakub Bohuš with generated id 11
Inserted user Emma Lujza Barilová with generated id 12
Inserted user SDA registrácia with generated id 13
Inserted user Jakub Bohuš with generated id 14
Inserted user Tobias Valúch with generated id 15
Inserted user Katarína Kšenzakovičová with generated id 16
Inserted user Barbora Sisková with generated id 17
Inserted user Natália  Zemanová  with generated id 18
Inserted user Tomáš Talárovič with generated id 19
Inserted user Sebastian Grof wit

Error inserting user null null: error: null value in column "email" of relation "users" violates not-null constraint
    at pe.Ue (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/chunk-2BOC2OMW.js:442:11)
    at pe.tt (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/chunk-2BOC2OMW.js:220:16)
    at pe.parse (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/chunk-2BOC2OMW.js:104:21)
    at N.qe (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/index.js:753:26)
    at file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/index.js:706:14
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x5a5ccf)
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x27eaeb)
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x2869d1)
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x286c2d)
    a

Inserted user Ester  Molitorisová with generated id 59
Inserted user Hana Hagarová with generated id 60
Inserted user Ján Viktor Poliak with generated id 61
Inserted user Andrea Kotrčová with generated id 62
Inserted user Tomáš Bačo with generated id 63
Inserted user Karolína Šofranková with generated id 64
Inserted user Matej Chriašteľ with generated id 65
Inserted user Ema Donátová with generated id 66
Inserted user Alexej Sopčák with generated id 67
Inserted user Daniel Čollák with generated id 68
Inserted user Ema Kovalčíková with generated id 69
Inserted user Barbora Krčová with generated id 70
Inserted user Amy Pargáčová with generated id 71
Inserted user Karin Kurucova with generated id 72
Inserted user Matúš Boleček with generated id 73
Inserted user Šimon Šima with generated id 74
Inserted user Radka Dorniaková with generated id 75
Inserted user megafon 278 with generated id 76
Inserted user Alena Tekelová with generated id 77
Inserted user Gregor Jamrich with generated id 78


Error inserting user null Deniel: error: null value in column "name" of relation "users" violates not-null constraint
    at pe.Ue (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/chunk-2BOC2OMW.js:442:11)
    at pe.tt (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/chunk-2BOC2OMW.js:220:16)
    at pe.parse (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/chunk-2BOC2OMW.js:104:21)
    at N.qe (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/index.js:753:26)
    at file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/index.js:706:14
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x5a5ccf)
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x27eaeb)
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x2869d1)
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x286c2d)
    

Inserted user Valentína Vyrostková with generated id 92
Inserted user Matyas Szabó with generated id 93
Inserted user Jožko Murgaš with generated id 94
Inserted user Martin J with generated id 95


...

Inserted user Marek Šimunek with generated id 1218
Inserted user Teo Justinič with generated id 1219
Inserted user Lívia Rosiarová with generated id 1220
Inserted user Lukáš Osúch with generated id 1221
Inserted user Lukáš  Dolinaj with generated id 1222
Inserted user Silvio  Petranský  with generated id 1223
Inserted user Nina Bukvayová  with generated id 1224
Inserted user Lucia Babicová with generated id 1225
Inserted user Filip Nemčický with generated id 1226
Inserted user Marian Babic with generated id 1227
Inserted user Romana Gagačová with generated id 1228
Inserted user Lujza Zacharová with generated id 1229
Inserted user Tamara Čerešňová with generated id 1230
Inserted user Mark Kurakin with generated id 1231
Inserted user Jozef Jeffrey Michalec with generated id 1232
Inserted user Martin Ponížil with generated id 1233
Inserted user Ján Jursa with generated id 1234
Inserted user Juraj Turčan with generated id 1235
Inserted user Agáta Vilimová with generated id 1236
Inserted us

Error inserting user null null: error: null value in column "name" of relation "users" violates not-null constraint
    at pe.Ue (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/chunk-2BOC2OMW.js:442:11)
    at pe.tt (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/chunk-2BOC2OMW.js:220:16)
    at pe.parse (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/chunk-2BOC2OMW.js:104:21)
    at N.qe (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/index.js:753:26)
    at file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/index.js:706:14
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x5a5ccf)
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x27eaeb)
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x2869d1)
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x286c2d)
    at

Inserted user Adam Bielovský with generated id 1290
Inserted user Dana Farkas with generated id 1291
Inserted user Mathias Kubíček with generated id 1292
Inserted user Nikola Húsková with generated id 1293
Inserted user Ina Opartyová with generated id 1294
Inserted user Viktor Jencus with generated id 1295
Inserted user Timotej Oršula with generated id 1296
Inserted user Tamara Kristufkova with generated id 1297
Inserted user Radka  Pavelkova  with generated id 1298
Inserted user Dominik Pokorný with generated id 1299
Inserted user Sebastián Soroka with generated id 1300
Inserted user Zuzana Guzyova with generated id 1301
Inserted user Martin Pollák with generated id 1302
Inserted user Veronika Timoshchenko with generated id 1303
Inserted user Eunika Blažová with generated id 1304
Inserted user Timotej Maličký with generated id 1305
Inserted user Alexandra  Slováková with generated id 1306
Inserted user Matej Ďurina with generated id 1307
Inserted user Andrej Štec with generated id 130

Error inserting user null vengrinp: error: null value in column "name" of relation "users" violates not-null constraint
    at pe.Ue (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/chunk-2BOC2OMW.js:442:11)
    at pe.tt (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/chunk-2BOC2OMW.js:220:16)
    at pe.parse (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/chunk-2BOC2OMW.js:104:21)
    at N.qe (file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/index.js:753:26)
    at file:///home/juhas/.cache/deno/npm/registry.npmjs.org/@electric-sql/pglite/0.5.7/dist/index.js:706:14
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x5a5ccf)
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x27eaeb)
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x2869d1)
    at <anonymous> (wasm://wasm/0267bb86:wasm-function[0]:0x286c2d)
  

Inserted user Michal Šarkan with generated id 1368
Inserted user Klaudia Rybárová with generated id 1369
Inserted user Dominika Truhanová with generated id 1370
Inserted user Karina Bernáthová with generated id 1371
Inserted user Eleanor Klenovska with generated id 1372
Inserted user Laura Plešová with generated id 1373
Inserted user Park mládeže Košice Gymnázium with generated id 1374
Inserted user Samuel Čorba with generated id 1375
Inserted user Viktória Vráželová with generated id 1376
Inserted user Ailin Hanková with generated id 1377
Inserted user Soňa Kelemenová with generated id 1378
Inserted user Lucia Kubíny with generated id 1379
Inserted user Marián Kubíny with generated id 1380
Inserted user Boris Kovalcik with generated id 1381
Inserted user Ivana  Mikolasova with generated id 1382
Inserted user Jakub Kovalcik with generated id 1383
Inserted user Simon Hursan with generated id 1384
Inserted user Katarína  Filičková  with generated id 1385
Inserted user Adam Dolný with gen

In [11]:
// Close the pglite client
await client.close();
